Timetravel -- Going back to a previous state of the graph, modifying it, and re-running from that point.

In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

#  STATE 
class State(TypedDict):
    count: int

#  NODES 
def step1(state):
    print("Step 1 running")
    return {"count": state["count"] + 1}

def step2(state):
    print("Step 2 running")
    return {"count": state["count"] + 10}

#  BUILD GRAPH 
builder = StateGraph(State)

builder.add_node("step1", step1)
builder.add_node("step2", step2)

builder.set_entry_point("step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", END)

# CHECKPOINTER (REQUIRED FOR TIME TRAVEL)
memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

# SESSION ID 
config = {"configurable": {"thread_id": "session-1"}}


# RUN GRAPH (creates history)

print("\n=== FIRST RUN ===")
result = graph.invoke({"count": 0}, config=config)

print("\nFinal Result:", result)

print("\n=== TIME TRAVEL: GET STATE ===")

history = graph.get_state(config)

print("Stored State Snapshot:")
print(history.values)

print("\n=== MODIFY STATE ===")

state = history.values

# time travel edit
state["count"] = 100

print("Modified State:", state)

print("\n=== RESUME AFTER TIME TRAVEL ===")

result2 = graph.invoke(state, config=config)

print("\nFinal After Time Travel:", result2)


=== FIRST RUN ===
Step 1 running
Step 2 running

Final Result: {'count': 11}

=== TIME TRAVEL: GET STATE ===
Stored State Snapshot:
{'count': 11}

=== MODIFY STATE ===
Modified State: {'count': 100}

=== RESUME AFTER TIME TRAVEL ===
Step 1 running
Step 2 running

Final After Time Travel: {'count': 111}
